In [ ]:
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_chroma import Chroma
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# 1. Setup Models
# Make sure your database was built with this SAME embedding model
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
persistent_directory = "db1/chroma_db"

db = Chroma(persist_directory=persistent_directory, embedding_function=embedding_model)


model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    temperature=0,
    max_retries=6,  # Automatically retries if you hit a rate limit
    convert_system_message_to_human=True # Better compatibility for Gemini
)

# 2. Conversation History
chat_history = []

def ask_question(user_question):
    global chat_history
    print(f"\n--- Processing: {user_question} ---")
    
    # STEP 1: Contextualize the question
    # This turns "How much did they pay?" into "How much did Microsoft pay for GitHub?"
    search_question = user_question
    if chat_history:
        context_prompt = [
            SystemMessage(content="Given the chat history and a new question, rewrite it as a standalone question that can be understood without the history. Just return the text of the new question."),
        ] + chat_history + [HumanMessage(content=f"Rewrite this question: {user_question}")]
        
        result = model.invoke(context_prompt)
        search_question = result.content.strip()
        print(f"Standalone Search Query: {search_question}")

    # STEP 2: Retrieve Documents
    retriever = db.as_retriever(search_kwargs={"k": 3})
    docs = retriever.invoke(search_question)
    
    # STEP 3: Generate Answer with Context
    context_text = "\n".join([f"- {doc.page_content}" for doc in docs])
    combined_input = f"""Answer the question using ONLY the provided documents.
    
    Documents:
    {context_text}
    
    Question: {user_question}
    """

    messages = [
        SystemMessage(content="You are a helpful assistant. Use the provided context to answer questions accurately."),
    ] + chat_history + [HumanMessage(content=combined_input)]
    
    response = model.invoke(messages)
    answer = response.content
    
    # STEP 4: Update History
    chat_history.append(HumanMessage(content=user_question))
    chat_history.append(AIMessage(content=answer))
    
    # Keep history manageable (last 6 messages)
    if len(chat_history) > 10:
        chat_history = chat_history[-10:]
        
    print(f"\nAnswer: {answer}")

def start_chat():
    print("Welcome to your RAG Chat! (Type 'quit' to exit)")
    while True:
        user_input = input("\nYour question: ")
        if user_input.lower() == 'quit':
            break
        ask_question(user_input)

if __name__ == "__main__":
    start_chat()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1734.96it/s]


Welcome to your RAG Chat! (Type 'quit' to exit)

--- Processing: What was Microsoft's first hardware product release ---


ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 2.079349207s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTi

In [ ]:
# Synthetic Questions: 

# 1. "What was NVIDIA's first graphics accelerator called?"
# 2. "Which company did NVIDIA acquire to enter the mobile processor market?"
# 3. "What was Microsoft's first hardware product release?"
# 4. "How much did Microsoft pay to acquire GitHub?"
# 5. "In what year did Tesla begin production of the Roadster?"
# 6. "Who succeeded Ze'ev Drori as CEO in October 2008?"
# 7. "What was the name of the autonomous spaceport drone ship that achieved the first successful sea landing?"
# 8. "What was the original name of Microsoft before it became Microsoft?"